# 09 — Model Persistence & Inference Pipeline

This notebook prepares the fraud-detection system for deployment.

The objectives are to:

- save the final CatBoost model,
- save the Platt probability calibrator,
- save the frozen risk thresholds,
- build a reusable inference function,
- verify that saved artifacts reproduce notebook predictions,
- prepare the project for Streamlit deployment.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

data_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fraud_modeling_dataset.parquet"
)

In [6]:
df = pd.read_parquet(data_path)

target = "fraud_bool"

train_df = df[
    df["month"].between(0, 4)
].copy()

calibration_df = df[
    df["month"] == 5
].copy()

strategy_df = df[
    df["month"] == 6
].copy()

test_df = df[
    df["month"] == 7
].copy()

In [ ]:
#prepare data for CatBoost model

cat_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os"
]

def prepare_catboost_data(X):
    X_cb = X.copy()

    for col in cat_features:
        X_cb[col] = (
            X_cb[col]
            .fillna("Missing")
            .astype(str)
        )

    return X_cb

In [7]:
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_cal = calibration_df.drop(columns=[target])
y_cal = calibration_df[target]

X_strategy = strategy_df.drop(columns=[target])
y_strategy = strategy_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

X_train_cb = prepare_catboost_data(
    X_train
)

X_cal_cb = prepare_catboost_data(
    X_cal
)

X_strategy_cb = prepare_catboost_data(
    X_strategy
)

X_test_cb = prepare_catboost_data(
    X_test
)

In [8]:
final_cat_model = CatBoostClassifier(
    depth=5,
    learning_rate=0.05,
    iterations=631,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False
)

final_cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features
)

CatBoostClassifier(auto_class_weights='Balanced', depth=5, iterations=631, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [ ]:
#Rebuild the Plattcalibrator

eps = 1e-6

cal_raw_proba = final_cat_model.predict_proba(
    X_cal_cb
)[:, 1]

cal_raw_clipped = np.clip(
    cal_raw_proba,
    eps,
    1 - eps
)

cal_logits = np.log(
    cal_raw_clipped
    / (1 - cal_raw_clipped)
).reshape(-1, 1)

platt_calibrator = LogisticRegression()

platt_calibrator.fit(
    cal_logits,
    y_cal
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [11]:
#Recreate the frozen month-6 risk thresholds

strategy_raw_proba = final_cat_model.predict_proba(
    X_strategy_cb
)[:, 1]

strategy_raw_clipped = np.clip(
    strategy_raw_proba,
    eps,
    1 - eps
)

strategy_logits = np.log(
    strategy_raw_clipped
    / (1 - strategy_raw_clipped)
).reshape(-1, 1)

strategy_calibrated_proba = (
    platt_calibrator
    .predict_proba(strategy_logits)[:, 1]
)

medium_threshold = np.quantile(
    strategy_calibrated_proba,
    0.80
)

high_threshold = np.quantile(
    strategy_calibrated_proba,
    0.95
)

critical_threshold = np.quantile(
    strategy_calibrated_proba,
    0.99
)

selected_review_threshold = np.quantile(
    strategy_calibrated_proba,
    0.95
)

print("Medium:", medium_threshold)
print("High:", high_threshold)
print("Critical:", critical_threshold)
print(
    "Review threshold:",
    selected_review_threshold
)

Medium: 0.012673651616958732
High: 0.05820484107195065
Critical: 0.19175058691953206
Review threshold: 0.05820484107195065


## 2. Save Model Artifacts

The trained CatBoost model, probability calibrator, and frozen decision thresholds are saved for reuse during deployment.

Keeping these artifacts separate allows the deployment application to reproduce the same scoring and decision logic used in the analytical notebooks.

In [12]:
catboost_model_path = (
    MODELS_DIR
    / "catboost_fraud_model.cbm"
)

final_cat_model.save_model(
    str(catboost_model_path)
)

print(
    "Saved:",
    catboost_model_path
)

Saved: D:\bank-account-fraud-decision-system\models\catboost_fraud_model.cbm


In [13]:
calibrator_path = (
    MODELS_DIR
    / "platt_calibrator.joblib"
)

joblib.dump(
    platt_calibrator,
    calibrator_path
)

print(
    "Saved:",
    calibrator_path
)

Saved: D:\bank-account-fraud-decision-system\models\platt_calibrator.joblib


In [14]:
thresholds = {
    "medium_threshold":
        float(medium_threshold),

    "high_threshold":
        float(high_threshold),

    "critical_threshold":
        float(critical_threshold),

    "review_threshold":
        float(selected_review_threshold)
}

threshold_path = (
    MODELS_DIR
    / "risk_thresholds.json"
)

with open(
    threshold_path,
    "w"
) as f:

    json.dump(
        thresholds,
        f,
        indent=4
    )

print(
    "Saved:",
    threshold_path
)

Saved: D:\bank-account-fraud-decision-system\models\risk_thresholds.json


In [15]:
metadata = {
    "target": "fraud_bool",

    "categorical_features":
        cat_features,

    "training_months": [
        0, 1, 2, 3, 4
    ],

    "calibration_month": 5,

    "strategy_month": 6,

    "final_test_month": 7,

    "model_type":
        "CatBoostClassifier",

    "iterations": 631,

    "depth": 5,

    "learning_rate": 0.05,

    "calibration_method":
        "Platt scaling",

    "primary_review_policy":
        "Top-risk applications above frozen month-6 5% threshold"
}

metadata_path = (
    MODELS_DIR
    / "model_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print(
    "Saved:",
    metadata_path
)

Saved: D:\bank-account-fraud-decision-system\models\model_metadata.json


In [16]:
for file in MODELS_DIR.iterdir():
    print(file.name)

catboost_fraud_model.cbm
model_metadata.json
platt_calibrator.joblib
risk_thresholds.json


## 3. Reusable Inference Function

The deployment pipeline should accept a single application record, reproduce the same preprocessing used during training, generate the CatBoost score, calibrate it with the saved Platt model, assign a risk band, and return a review recommendation.

In [18]:
#Reload the model, calibrator, and thresholds for inference

from catboost import CatBoostClassifier
import joblib
import json
import numpy as np
import pandas as pd

loaded_model = CatBoostClassifier()
loaded_model.load_model(
    str(MODELS_DIR / "catboost_fraud_model.cbm")
)

loaded_calibrator = joblib.load(
    MODELS_DIR / "platt_calibrator.joblib"
)

with open(
    MODELS_DIR / "risk_thresholds.json",
    "r"
) as f:
    loaded_thresholds = json.load(f)

with open(
    MODELS_DIR / "model_metadata.json",
    "r"
) as f:
    loaded_metadata = json.load(f)

loaded_thresholds

{'medium_threshold': 0.012673651616958732,
 'high_threshold': 0.05820484107195065,
 'critical_threshold': 0.19175058691953206,
 'review_threshold': 0.05820484107195065}

In [19]:
#Create the inference function

def predict_fraud_risk(application_data):
    """
    Predict calibrated fraud risk for one application.

    Parameters
    ----------
    application_data : dict
        One application containing the same model features
        used during training.

    Returns
    -------
    dict
        Raw CatBoost score, calibrated probability,
        risk band, and review recommendation.
    """

    application = pd.DataFrame(
        [application_data]
    )

    # Preserve exact training feature order
    application = application[
        X_train.columns
    ].copy()

    # Prepare categorical variables
    for col in loaded_metadata[
        "categorical_features"
    ]:
        application[col] = (
            application[col]
            .fillna("Missing")
            .astype(str)
        )

    # Raw CatBoost probability
    raw_probability = (
        loaded_model
        .predict_proba(application)[0, 1]
    )

    # Convert to logit for Platt calibration
    clipped_probability = np.clip(
        raw_probability,
        1e-6,
        1 - 1e-6
    )

    logit_score = np.log(
        clipped_probability
        / (1 - clipped_probability)
    )

    calibrated_probability = (
        loaded_calibrator
        .predict_proba(
            np.array(
                [[logit_score]]
            )
        )[0, 1]
    )

    # Risk segmentation
    if calibrated_probability >= (
        loaded_thresholds[
            "critical_threshold"
        ]
    ):
        risk_band = "Critical"

    elif calibrated_probability >= (
        loaded_thresholds[
            "high_threshold"
        ]
    ):
        risk_band = "High"

    elif calibrated_probability >= (
        loaded_thresholds[
            "medium_threshold"
        ]
    ):
        risk_band = "Medium"

    else:
        risk_band = "Low"

    review_required = (
        calibrated_probability
        >= loaded_thresholds[
            "review_threshold"
        ]
    )

    return {
        "raw_model_score":
            float(raw_probability),

        "calibrated_probability":
            float(calibrated_probability),

        "risk_band":
            risk_band,

        "manual_review":
            bool(review_required)
    }

In [21]:
#Test the function using one actual month-7 application

example_application = (
    X_test
    .iloc[0]
    .to_dict()
)

prediction = predict_fraud_risk(
    example_application
)

prediction

{'raw_model_score': 0.08565911574863047,
 'calibrated_probability': 0.0017592034012743896,
 'risk_band': 'Low',
 'manual_review': False}

In [25]:
test_raw_proba = loaded_model.predict_proba(
    X_test_cb
)[:, 1]

test_raw_clipped = np.clip(
    test_raw_proba,
    1e-6,
    1 - 1e-6
)

test_logits = np.log(
    test_raw_clipped
    / (1 - test_raw_clipped)
).reshape(-1, 1)

test_calibrated_proba = (
    loaded_calibrator
    .predict_proba(test_logits)[:, 1]
)

original_probability = (
    test_calibrated_proba[0]
)

print(
    "Loaded pipeline probability:",
    prediction[
        "calibrated_probability"
    ]
)

print(
    "Original notebook probability:",
    original_probability
)

print(
    "Absolute difference:",
    abs(
        prediction[
            "calibrated_probability"
        ]
        - original_probability
    )
)

Loaded pipeline probability: 0.0017592034012743896
Original notebook probability: 0.0017592034012743896
Absolute difference: 0.0


## 4. Deployment Consistency Check

The saved CatBoost model, Platt calibrator, and frozen decision thresholds were reloaded and tested on a month-7 application.

The calibrated fraud probability produced by the reloaded inference pipeline exactly matched the original notebook result, with an absolute difference of 0.0.

This confirms that the persisted artifacts reproduce the analytical pipeline correctly and are ready for use in a deployment application.

In [12]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

data_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fraud_modeling_dataset.parquet"
)

df = pd.read_parquet(data_path)

target = "fraud_bool"

train_df = df[
    df["month"].between(0, 4)
].copy()

X_train = train_df.drop(
    columns=[target]
)

y_train = train_df[target]

print(X_train.shape)

(675666, 40)


In [13]:
raw_input_features = [
    "income",
    "name_email_similarity",
    "prev_address_months_count",
    "current_address_months_count",
    "customer_age",
    "days_since_request",
    "intended_balcon_amount",
    "payment_type",
    "zip_count_4w",
    "velocity_6h",
    "velocity_24h",
    "velocity_4w",
    "bank_branch_count_8w",
    "date_of_birth_distinct_emails_4w",
    "employment_status",
    "credit_risk_score",
    "email_is_free",
    "housing_status",
    "phone_home_valid",
    "phone_mobile_valid",
    "bank_months_count",
    "has_other_cards",
    "proposed_credit_limit",
    "foreign_request",
    "source",
    "session_length_in_minutes",
    "device_os",
    "keep_alive_session",
    "device_distinct_emails_8w",
    "month"
]

In [14]:
cat_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os"
]

raw_input_features = [
    "income",
    "name_email_similarity",
    "prev_address_months_count",
    "current_address_months_count",
    "customer_age",
    "days_since_request",
    "intended_balcon_amount",
    "payment_type",
    "zip_count_4w",
    "velocity_6h",
    "velocity_24h",
    "velocity_4w",
    "bank_branch_count_8w",
    "date_of_birth_distinct_emails_4w",
    "employment_status",
    "credit_risk_score",
    "email_is_free",
    "housing_status",
    "phone_home_valid",
    "phone_mobile_valid",
    "bank_months_count",
    "has_other_cards",
    "proposed_credit_limit",
    "foreign_request",
    "source",
    "session_length_in_minutes",
    "device_os",
    "keep_alive_session",
    "device_distinct_emails_8w",
    "month"
]

In [15]:
numeric_input_features = [
    col for col in raw_input_features
    if col not in cat_features
]

feature_schema = {
    "raw_input_features": raw_input_features,
    "categorical_features": cat_features,
    "model_features": X_train.columns.tolist(),
    "defaults": {}
}

for col in numeric_input_features:

    series = X_train[col]

    valid_values = series[
        series.notna()
    ]

    feature_schema["defaults"][col] = (
        float(valid_values.median())
    )

In [17]:
feature_schema["categorical_options"] = {
    "payment_type": [
        "AA", "AB", "AC", "AD", "AE"
    ],

    "employment_status": [
        "CA", "CB", "CC", "CD",
        "CE", "CF", "CG"
    ],

    "housing_status": [
        "BA", "BB", "BC", "BD",
        "BE", "BF", "BG"
    ],

    "source": [
        "INTERNET",
        "TELEAPP"
    ],

    "device_os": [
        "linux",
        "macintosh",
        "other",
        "windows",
        "x11"
    ]
}

In [20]:
from pathlib import Path
import json

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(MODELS_DIR)

D:\bank-account-fraud-decision-system\models


In [22]:
schema_path = (
    MODELS_DIR
    / "feature_schema.json"
)

with open(
    schema_path,
    "w"
) as f:

    json.dump(
        feature_schema,
        f,
        indent=4
    )

print("Saved:", schema_path)

Saved: D:\bank-account-fraud-decision-system\models\feature_schema.json


In [23]:
print(schema_path.exists())

True
